# Explainability Analysis

## Objective

This notebook examines the linguistic features influencing machine learning predictions. The goal is to connect computational classification with literary interpretation by identifying words and patterns associated with therapeutic representation.

In [2]:
from google.colab import files

uploaded = files.upload()

Saving frankenstein_training_dataset_v2.csv to frankenstein_training_dataset_v2.csv


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "frankenstein_training_dataset_v2.csv"
)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully
Dataset shape: (39, 7)


,passage_id,text,therapeutic_label,mechanism,intensity,annotator_notes,keyword_score
0,10,This appearance excited our unqualified wonder...,0.0,NaN,0.0,The passage describes curiosity and wonder dur...,NaN
1,16,Will you smile at the enthusiasm I express con...,1.0,Narrative Reconstruction,3.0,The passage introduces Victor's suffering and ...,NaN
2,26,I feel exquisite pleasure in dwelling on the r...,1.0,Narrative Reconstruction,4.0,The passage represents autobiographical reflec...,NaN
3,67,"“She most of all,” said Ernest, “requires cons...",0.0,NaN,0.0,"The passage describes grief, guilt, and emotio...",NaN
4,127,“I knocked. ‘Who is there?’ said the old man. ...,0.0,NaN,0.0,The passage represents a compassionate social ...,NaN


## TF-IDF Feature Reconstruction

The annotated passages are transformed into TF-IDF vectors to identify the linguistic features contributing to machine learning predictions.

The extracted feature weights will be examined to understand which words influence therapeutic classification.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=1000,
    stop_words="english"
)

X = tfidf.fit_transform(
    df["text"]
)

y = df["therapeutic_label"]

feature_names = tfidf.get_feature_names_out()

print("TF-IDF recreated successfully")
print("Feature matrix shape:", X.shape)
print("Number of features:", len(feature_names))

TF-IDF recreated successfully
Feature matrix shape: (39, 1000)
Number of features: 1000


## Logistic Regression Feature Importance

Logistic Regression coefficients are analyzed to identify the words that contribute most strongly toward therapeutic and non-therapeutic classifications.

This provides an interpretable connection between machine learning predictions and literary features.

In [5]:
from sklearn.linear_model import LogisticRegression

explain_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

explain_model.fit(
    X,
    y
)

print("Explainability model trained successfully")

Explainability model trained successfully


In [6]:
importance = explain_model.coef_[0]

feature_importance = pd.DataFrame({
    "word": feature_names,
    "importance": importance
})

feature_importance.head()

,word,importance
0,17,-0.022148
1,19th,0.052280
2,2d,0.049994
3,_ennui_,-0.056768
4,_i,-0.079938


In [7]:
positive_words = feature_importance.sort_values(
    by="importance",
    ascending=False
).head(15)

positive_words

,word,importance
596,mind,0.256488
342,feel,0.242915
204,dear,0.239144
871,tale,0.235417
942,voice,0.220975
623,nature,0.212935
531,knowledge,0.202908
184,creation,0.200980
466,heaven,0.192860
615,mountains,0.191951


In [8]:
negative_words = feature_importance.sort_values(
    by="importance",
    ascending=True
).head(15)

negative_words

,word,importance
599,miserable,-0.275490
325,eyes,-0.247724
631,night,-0.234447
797,shall,-0.212367
526,justine,-0.210573
738,remorse,-0.205076
963,william,-0.188529
239,died,-0.184771
763,room,-0.178135
680,poor,-0.178102


## Logistic Regression Feature Importance Findings

The feature analysis revealed that therapeutic classifications were associated with linguistic markers of reflection, emotional connection, and meaning-making. Words such as "mind", "feel", "tale", "friend", and "nature" contributed positively toward therapeutic classification.

Non-therapeutic classifications were associated with vocabulary related to loss, guilt, and suffering, including "miserable", "died", "remorse", "Justine", and "William".

An important observation was that emotionally negative words such as "despair" were not automatically classified as non-therapeutic. This suggests that the model learned contextual patterns where suffering combined with reflection and reconstruction may represent therapeutic processing.

In [9]:
feature_importance.to_csv(
    "logistic_feature_importance_batch02.csv",
    index=False
)

print("Feature importance saved")

Feature importance saved


In [10]:
from google.colab import files

files.download(
    "logistic_feature_importance_batch02.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X,
    y,
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Split recreated")
print("Test samples:", len(idx_test))

Split recreated
Test samples: 8


In [12]:
from sklearn.linear_model import LogisticRegression

error_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

error_model.fit(
    X_train,
    y_train
)

error_predictions = error_model.predict(X_test)

print("Model retrained")

Model retrained


In [13]:
error_analysis = df.loc[idx_test, [
    "passage_id",
    "text",
    "therapeutic_label"
]].copy()

error_analysis["predicted_label"] = error_predictions

error_analysis["correct"] = (
    error_analysis["therapeutic_label"]
    ==
    error_analysis["predicted_label"]
)

error_analysis

,passage_id,text,therapeutic_label,predicted_label,correct
13,125,"“Several changes, in the meantime, took place ...",0.0,0.0,True
24,151,It was in the latter end of September that I a...,0.0,1.0,False
20,78,"I could not answer. “No, Justine,” said Elizab...",1.0,0.0,False
29,62,Clerval spoke thus as we hurried through the s...,1.0,1.0,True
12,79,And on the morrow Justine died. Elizabeth’s he...,0.0,0.0,True
2,26,I feel exquisite pleasure in dwelling on the r...,1.0,1.0,True
19,81,"This advice, although good, was totally inappl...",0.0,0.0,True
1,16,Will you smile at the enthusiasm I express con...,1.0,1.0,True


In [14]:
misclassified = error_analysis[
    error_analysis["correct"] == False
]

misclassified

,passage_id,text,therapeutic_label,predicted_label,correct
24,151,It was in the latter end of September that I a...,0.0,1.0,False
20,78,"I could not answer. “No, Justine,” said Elizab...",1.0,0.0,False


## Error Analysis Findings

The Logistic Regression classifier produced two misclassifications among the eight test passages.

Passage 151 was incorrectly classified as therapeutic. The model appears to have responded to reflective and emotional vocabulary, although human annotation identified the passage as non-therapeutic because it represents distress and psychological decline rather than healing.

Passage 78 was incorrectly classified as non-therapeutic despite receiving a therapeutic annotation. The model likely relied on suffering-related vocabulary and failed to recognize therapeutic elements involving compassion, emotional support, and acknowledgement of innocence.

These errors demonstrate that computational models can identify linguistic patterns but may struggle to distinguish between emotional suffering and therapeutic transformation. Human annotation remains essential for interpreting complex literary representations of healing.

In [15]:
error_analysis.to_csv(
    "error_analysis_batch02.csv",
    index=False
)

print("Error analysis saved")

Error analysis saved


In [16]:
from google.colab import files

files.download(
    "error_analysis_batch02.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>